# 04 : GRU global de 16 salidas

Una muestra es una tienda–familia en un origen. Su objetivo contiene las ventas de los 16 días siguientes, emitidas juntas; no son 1.782 salidas simultáneas ni una predicción recursiva que realimente cada día. Los pesos se comparten entre series. El modo predeterminado es `full`, con datos reales y CUDA; `smoke` comprueba funcionamiento con datos sintéticos.

## Ejecución de la entrega

Ejecuta todas las celdas en orden en una sesión limpia. El modo predeterminado
`full` utiliza datos reales; el notebook 04 requiere GPU. Para comprobar solo
esquema y ejecución, define `os.environ["RETAIL_FORECAST_MODE"] = "smoke"` antes
de la preparación. Los resultados sintéticos no miden precisión.

Mantén los siete CSV en `data/raw` del espacio de trabajo o usa el ZIP de Kaggle.
No escribas credenciales en celdas ni salidas. Conserva el notebook ejecutado y
el directorio de resultados antes de desconectar Colab.

In [ ]:
# Immutable package setup; see docs/01_reproducibility.md for clean local mode.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip() or globals().get("REPO_URL", "")
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "4a6748b8143ef09c83d899576f98c9572d934ee1"
MODELS = False
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB, neural=True
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
WORKSPACE_ROOT = globals().get(
    "WORKSPACE_ROOT",
    Path("/content") if IN_COLAB else (
        Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else REPO_ROOT
    ),
)
os.chdir(REPO_ROOT)

In [ ]:
from dataclasses import replace

import json

import numpy as np
import pandas as pd
from IPython.display import display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import read_completed_run
from retail_forecast.config import load_config
from retail_forecast.data import required_files_exist, validate_dataset, write_synthetic_dataset
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.pipeline import predict, request
from retail_forecast.recurrent import recurrent_experiment
from retail_forecast.recurrent_feasibility import recurrent_feasibility

MODE = os.getenv("RETAIL_FORECAST_MODE", "full")
if MODE not in {"smoke", "full"}:
    raise ValueError("RETAIL_FORECAST_MODE must be smoke or full")
config = load_config(
    REPO_ROOT / ("configs/gru.yaml" if MODE == "full" else "configs/gru-smoke.yaml")
)
config = replace(
    config,
    paths=replace(
        config.paths,
        raw_data_dir=WORKSPACE_ROOT / config.paths.raw_data_dir.relative_to(REPO_ROOT),
        artifacts_dir=WORKSPACE_ROOT / config.paths.artifacts_dir.relative_to(REPO_ROOT),
    ),
)
print("Mode:", MODE, "Policy:", config.data.availability_policy)
print("Recipe:", config.recurrent)
print("Runtime identity:", runtime_evidence)

## Datos e información disponible

Se reutilizan los CSV o el ZIP. La receta completa usa `asof_v1`: ventas y transacciones hasta el origen, sin petróleo ni filas de calendario de tipo `Event`. Promociones y otros festivos aproximan información planificada, sin versiones históricas que permitan verificarla por completo.

In [ ]:
if config.data.source == "synthetic":
    write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir, (WORKSPACE_ROOT / "data", WORKSPACE_ROOT), token_reader=getpass
    )
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
display(pd.DataFrame([validate_dataset(config).to_dict()]))

## Arquitectura, cortes y normalización

El lote tiene historia `[B, 56, 8]`, covariables futuras admitidas `[B, 16, 12]`, seis códigos categóricos y objetivo `[B, 16]`. Una capa GRU resume la historia; su estado, embeddings y covariables pasan a la cabeza de 16 salidas.

Para cada serie s se guardan media μ_s y desviación poblacional σ_s de `log1p(ventas)` sobre las fechas elegibles de entrenamiento. Se aplica `z = (log1p(y) - μ_s) / σ_s`; una desviación menor que 1e-6 se sustituye por uno. Promociones y transacciones tienen estadísticas propias. Los ausentes se rellenan en escala transformada con máscara separada. Estandarizar no demuestra estacionariedad.

En origen externo O, el corte interno C=O−16 selecciona épocas en C+1…O. Después se crean pesos y transformaciones nuevos hasta O, antes de evaluar O+1…O+16. La pérdida deshace la estandarización del objetivo y calcula MSE en escala logarítmica original. La prueba de recursos no calcula errores externos.

In [ ]:
probe = recurrent_feasibility(config)
probe_summary = json.loads(Path(probe["summary"]).read_text())
display(
    pd.DataFrame(
        [
            {
                k: probe_summary[k]
                for k in (
                    "phase",
                    "outer_scores_computed",
                    "device_name",
                    "host_peak_bytes",
                    "wall_seconds",
                    "estimated_max_primary_seconds",
                )
            }
        ]
    )
)
print("Probe evidence:", probe["run_dir"])

In [ ]:
study = recurrent_experiment(config)
read_completed_run(Path(study["run_dir"]))
summary = json.loads(Path(study["summary"]).read_text())
metrics = pd.read_csv(Path(study["run_dir"]) / "metrics/recurrent_results.csv")
oof = pd.read_parquet(study["oof"])
assert len(oof) == summary["oof_rows"] == summary["expected_rows"]
assert oof["lineage_status"].eq("complete").all()
print("Study evidence:", study["run_dir"])
display(
    pd.DataFrame(
        [
            {
                k: summary[k]
                for k in (
                    "empirical_results_are_synthetic",
                    "oof_rows",
                    "rmsle",
                    "mae",
                    "signed_bias",
                    "wape",
                    "primary_training_seconds",
                )
            }
        ]
    )
)
display(metrics.loc[metrics["scope"].isin(["fold", "horizon"])])

## Evolución del entrenamiento

La métrica agrupada es la principal. Las historias muestran cuántas épocas se probaron internamente y cuántas se usaron en el reajuste; no se eligen épocas con los errores externos.

In [ ]:
selections = json.loads((Path(study["run_dir"]) / "metrics/recurrent_selection.json").read_text())
histories = []
for selected in selections:
    for stage in selected["stages"]:
        histories.extend(
            {"fold": selected["fold"], "stage": stage["stage"], **row} for row in stage["history"]
        )
display(pd.DataFrame(histories))
print("Pooled OOF RMSLE:", summary["rmsle"])
print("Mean fold RMSLE (descriptive):", metrics.loc[metrics["scope"].eq("fold"), "rmsle"].mean())
print("Resource evidence:", Path(study["run_dir"]) / "metrics/recurrent_costs.json")

## Una muestra nueva usa el estado guardado

Una ventana posterior de una serie conocida se transforma con los mismos μ_s, σ_s y vocabularios del modelo, sin reajustar estadísticas sobre esa ventana. La inversión es `max(0, expm1(μ_s + σ_s * z_pred))`. Una tienda–familia desconocida se rechaza; actualizar el escalador requiere entrenar una versión nueva.

La siguiente celda recarga el primer modelo y reproduce su origen para comprobar paridad de predicción. Esta prueba de recarga no equivale a una nueva evaluación independiente ni afirma haber probado una tienda desconocida.

In [ ]:
bundle = f"{study['training_runs'][0]}:models/gru_global/fold_1"
normalized = request(config, origin=config.validation.origins[0])
reloaded = predict(config, bundle=bundle, request_dir=Path(normalized["request_dir"]))
forecast = pd.read_csv(reloaded["forecast"])
reference = oof.loc[
    oof["fold"].eq("fold_1"), ["store_nbr", "family", "target_date", "y_pred"]
].copy()
reference["target_date"] = pd.to_datetime(reference["target_date"]).dt.strftime("%Y-%m-%d")
matched = forecast.merge(
    reference, on=["store_nbr", "family", "target_date"], validate="one_to_one"
)
assert len(matched) == len(forecast) == len(reference)
np.testing.assert_allclose(matched["prediction"], matched["y_pred"], rtol=1e-5, atol=1e-4)
print("Frozen-state parity rows:", len(matched))
print("Prediction evidence:", reloaded["run_dir"])
display(forecast.head())

## Interpretación de la ejecución real registrada

La GRU nueva obtuvo RMSLE 0,467408 frente a 0,398680 de LightGBM. En el día 16 alcanzó 0,520109, frente a 0,499748 del promedio de cuatro semanas: una métrica agrupada cercana oculta diferencias por horizonte. En SCHOOL AND OFFICE SUPPLIES la GRU obtuvo 0,853456 y LightGBM 0,658960. Los CSV conservan los resultados completos.

Estas cifras pertenecen a la ejecución Colab registrada en `results/colab_execution.json`; el modo sintético no las reproduce. Tres ventanas cercanas no establecen estabilidad anual ni beneficios de inventario. El siguiente notebook estudia una inicialización transferida con otra receta y solo sobre tiendas destino.